# 04 — Existing Chargers Baseline

Load NAP charging point dataset. Filter to stations on interurban roads only. Calculate `total_existing_stations_baseline` for File 1. Identify current coverage gaps.

## Data Inputs
- `data/processed/chargers_clean.csv`
- `data/processed/interurban_roads.geojson`

## Data Outputs
- `data/processed/interurban_chargers.csv` — existing chargers on interurban roads
- `total_existing_stations_baseline` scalar (for File_1.csv)
- `data/interim/coverage_gaps.geojson` — uncovered road segments

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
from src.constants import POWER_PER_CHARGER_KW

PROCESSED = Path('data/processed')
INTERIM = Path('data/interim')
INTERIM.mkdir(parents=True, exist_ok=True)

# Load datasets
chargers = pd.read_csv(PROCESSED / 'chargers_clean.csv')
roads = gpd.read_file(PROCESSED / 'interurban_roads.geojson')

print(f"Total NAP chargers: {chargers.shape[0]}")
print(f"Interurban road segments: {roads.shape[0]}")
print(f"\nCharger columns: {list(chargers.columns)}")
chargers.head()

## Filter Chargers to Interurban Roads

Buffer interurban roads by 2 km. Any charger within this buffer is considered "on" an interurban corridor.

In [ ]:
# Create GeoDataFrame from chargers
chargers_gdf = gpd.GeoDataFrame(
    chargers,
    geometry=gpd.points_from_xy(chargers['longitude'], chargers['latitude']),
    crs='EPSG:4326'
)

# Project to metric CRS for buffering (EPSG:25830 = ETRS89 UTM Zone 30N, covers Spain)
roads_utm = roads.to_crs('EPSG:25830')
chargers_utm = chargers_gdf.to_crs('EPSG:25830')

# Buffer roads by 2 km
BUFFER_M = 2000
road_buffer = roads_utm.geometry.buffer(BUFFER_M).union_all()
print(f"Road buffer created: 2 km around {len(roads_utm)} interurban segments")

# Spatial filter: chargers within the road buffer
mask = chargers_utm.geometry.within(road_buffer)
interurban_chargers = chargers_gdf[mask].copy()

print(f"\nChargers near interurban roads: {len(interurban_chargers)} / {len(chargers_gdf)}")
print(f"  ({len(interurban_chargers)/len(chargers_gdf)*100:.1f}% of all NAP chargers)")

# Power stats
if 'max_power_kw' in interurban_chargers.columns:
    print(f"\nPower distribution (kW):")
    print(interurban_chargers['max_power_kw'].describe().round(1))
    fast = interurban_chargers['max_power_kw'] >= 50
    print(f"\nFast chargers (≥50 kW): {fast.sum()} ({fast.mean()*100:.1f}%)")
    ultrafast = interurban_chargers['max_power_kw'] >= 150
    print(f"Ultra-fast chargers (≥150 kW): {ultrafast.sum()} ({ultrafast.mean()*100:.1f}%)")

## Assign Chargers to Nearest Road Segment

Spatial join each charger to the nearest interurban road segment to identify which corridor it serves.

In [ ]:
# Spatial join: nearest road segment for each charger
interurban_chargers_utm = interurban_chargers.to_crs('EPSG:25830')
joined = gpd.sjoin_nearest(
    interurban_chargers_utm,
    roads_utm[['segment_id', 'Carretera', 'road_prefix', 'is_tent', 'geometry']],
    how='left',
    distance_col='dist_to_road_m'
)

# Add road info back
interurban_chargers['segment_id'] = joined['segment_id'].values
interurban_chargers['nearest_road'] = joined['Carretera'].values
interurban_chargers['road_prefix'] = joined['road_prefix'].values
interurban_chargers['is_tent'] = joined['is_tent'].values
interurban_chargers['dist_to_road_m'] = joined['dist_to_road_m'].values

print("Chargers per road type:")
print(interurban_chargers['road_prefix'].value_counts())
print(f"\nChargers on TEN-T corridors: {interurban_chargers['is_tent'].sum()}")
print(f"\nMean distance to road: {interurban_chargers['dist_to_road_m'].mean():.0f} m")

## Identify Coverage Gaps

Find road segments that are farther than the maximum spacing threshold from any existing charger. These are candidate locations for new stations.

In [ ]:
from src.constants import MAX_STATION_SPACING_KM, AFIR_SPACING_KM

# For each road segment, compute distance to nearest existing charger
# Use segment centroids for simplicity
seg_centroids = roads_utm.copy()
seg_centroids['geometry'] = seg_centroids.geometry.centroid

charger_points = interurban_chargers_utm[['geometry']].copy()

if len(charger_points) > 0:
    nearest = gpd.sjoin_nearest(
        seg_centroids[['segment_id', 'is_tent', 'max_spacing_km', 'geometry']],
        charger_points,
        how='left',
        distance_col='dist_to_charger_m'
    )
    # Take minimum distance per segment (in case of multiple matches)
    nearest_dist = nearest.groupby('segment_id')['dist_to_charger_m'].min().reset_index()
    roads_with_dist = roads.merge(nearest_dist, on='segment_id', how='left')

    # Convert to km
    roads_with_dist['dist_to_charger_km'] = roads_with_dist['dist_to_charger_m'] / 1000

    # A segment is a "gap" if distance to nearest charger exceeds its spacing threshold
    roads_with_dist['is_gap'] = (
        roads_with_dist['dist_to_charger_km'] > roads_with_dist['max_spacing_km']
    )

    gaps = roads_with_dist[roads_with_dist['is_gap']].copy()
    print(f"Coverage gaps: {len(gaps)} / {len(roads)} segments exceed spacing threshold")
    print(f"  TEN-T gaps (>{AFIR_SPACING_KM} km): {gaps['is_tent'].sum()}")
    print(f"  Non-TEN-T gaps (>{MAX_STATION_SPACING_KM} km): {(~gaps['is_tent']).sum()}")
    print(f"\nGap distance stats (km):")
    print(gaps['dist_to_charger_km'].describe().round(1))
else:
    roads_with_dist = roads.copy()
    roads_with_dist['dist_to_charger_km'] = np.nan
    roads_with_dist['is_gap'] = True
    gaps = roads_with_dist.copy()
    print("No interurban chargers found — all segments are gaps")

## Summary & Save Outputs

In [ ]:
# Baseline KPI for File 1
total_existing_stations_baseline = len(interurban_chargers)

print("=" * 60)
print("EXISTING CHARGERS BASELINE — SUMMARY")
print("=" * 60)
print(f"total_existing_stations_baseline = {total_existing_stations_baseline}")
print(f"\nBy road type:")
print(interurban_chargers['road_prefix'].value_counts().to_string())
print(f"\nCoverage gaps: {len(gaps)} segments need new chargers")

# Save interurban chargers CSV
drop_cols = ['geometry']
out_chargers = PROCESSED / 'interurban_chargers_baseline.csv'
interurban_chargers.drop(columns=drop_cols, errors='ignore').to_csv(out_chargers, index=False)
print(f"\nSaved: {out_chargers} ({len(interurban_chargers)} rows)")

# Save coverage gaps GeoJSON
out_gaps = INTERIM / 'coverage_gaps.geojson'
gaps.to_file(out_gaps, driver='GeoJSON')
print(f"Saved: {out_gaps} ({len(gaps)} segments)")

# Save baseline KPI as a small CSV for easy downstream use
pd.DataFrame([{
    'total_existing_stations_baseline': total_existing_stations_baseline,
}]).to_csv(PROCESSED / 'baseline_kpi.csv', index=False)
print(f"Saved: {PROCESSED / 'baseline_kpi.csv'}")